<a href="https://colab.research.google.com/github/Puspita02/AI-generated-Text-Detector/blob/main/01_Main_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content

!git clone https://github.com/Puspita02/Prompt-Bias-Mitigation-SLM-Judge.git

/content
Cloning into 'Prompt-Bias-Mitigation-SLM-Judge'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [3]:
%cd Prompt-Bias-Mitigation-SLM-Judge

/content/Prompt-Bias-Mitigation-SLM-Judge


In [4]:
import os

folders = [
    "datasets",
    "results",
    "figures",
    "logs",
    "prompts"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [5]:
!pip -q install transformers
!pip -q install datasets
!pip -q install accelerate

In [6]:
import torch
import transformers
import datasets
import huggingface_hub
import pandas
import numpy

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Torch: 2.11.0+cu128
Transformers: 5.12.1
Datasets: 4.0.0
HF Hub: 1.20.1
Pandas: 2.2.2
NumPy: 2.0.2


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("/content/Prompt-Bias-Mitigation-SLM-Judge")

DATA_DIR = PROJECT_ROOT / "datasets"
RESULT_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "figures"
LOG_DIR = PROJECT_ROOT / "logs"

DATA_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

print(PROJECT_ROOT)

/content/Prompt-Bias-Mitigation-SLM-Judge


In [8]:
from pathlib import Path

PROJECT_ROOT = Path("/content/Prompt-Bias-Mitigation-SLM-Judge")

RAW_DATA = PROJECT_ROOT / "datasets" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "datasets" / "processed"

for folder in [
    RAW_DATA / "mtbench",
    RAW_DATA / "llmbar",
    PROCESSED_DATA,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Dataset folders created successfully.")

Dataset folders created successfully.


In [9]:
from datasets import load_dataset

mtbench = load_dataset("lmsys/mt_bench_human_judgments")

mt_df = mtbench["gpt4_pair"].to_pandas()

mt_df.to_csv(
    RAW_DATA / "mtbench" / "mtbench.csv",
    index=False
)

print(mt_df.shape)


README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/gpt4_pair-00000-of-00001-c0b431264a(…):   0%|          | 0.00/650k [00:00<?, ?B/s]

data/human-00000-of-00001-25f49108187592(…):   0%|          | 0.00/739k [00:00<?, ?B/s]

Generating gpt4_pair split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating human split:   0%|          | 0/3355 [00:00<?, ? examples/s]

(2400, 8)


In [10]:
import os

%cd /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw

if not os.path.exists("LLMBar"):
    !git clone https://github.com/princeton-nlp/LLMBar.git

print("LLMBar repository ready.")

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw
Cloning into 'LLMBar'...
remote: Enumerating objects: 1725, done.
remote: Counting objects: 100% (1725/1725), done.
remote: Compressing objects: 100% (1227/1227), done.
remote: Total 1725 (delta 507), reused 1708 (delta 497), pack-reused 0 (from 0)
Receiving objects: 100% (1725/1725), 10.62 MiB | 12.24 MiB/s, done.
Resolving deltas: 100% (507/507), done.
LLMBar repository ready.


In [11]:
!ls -R /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar


/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar:
Dataset  LICENSE  LLMEvaluator	README.md  requirements.txt

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset:
CaseStudy  LLMBar  Processed

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy:
Base_10  Base_9  Constraint  Negation  Normal

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10:
dataset.json  evaluators

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators:
ChatGPT  ChatGPT-0301  Falcon  GPT-4  LLaMA2  PaLM2

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/ChatGPT:
CoT  Metrics_Reference	Swap  Swap_CoT	Vanilla

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/ChatGPT/CoT:
result.json  statistics.json

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Bas

In [12]:
!ls /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar

Dataset  LICENSE  LLMEvaluator	README.md  requirements.txt


In [13]:
!find /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset -type f

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/Metrics_Reference/statistics.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/Metrics_Reference/result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/Swap/statistics.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/Swap/result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/Swap_CoT/statistics.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/Swap_CoT/result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/evaluators/LLaMA2/CoT/statistics.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/CaseStudy/Base_10/ev

In [14]:
import os

dataset_dir = "/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset"

for item in sorted(os.listdir(dataset_dir)):
    print(item)

CaseStudy
LLMBar
Processed


In [15]:
import os

llmbar_path = "/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar"

print("Folders:")
for item in sorted(os.listdir(llmbar_path)):
    print(item)

Folders:
Adversarial
Natural


In [16]:
import os

llmbar_path = "/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar"

for subset in ["Natural", "Adversarial"]:
    print(f"\n===== {subset} =====")
    subset_path = os.path.join(llmbar_path, subset)

    for root, dirs, files in os.walk(subset_path):
        print(root)
        for f in files:
            print("   ", f)


===== Natural =====
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural
    dataset.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Vanilla_1shot
    statistics.json
    result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Metrics_Reference
    statistics.json
    result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Swap
    statistics.json
    result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Vanilla_NoRules
    statistics.json
    result.json
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset

In [19]:
import os

llmbar_root = "/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar"

for root, dirs, files in os.walk(llmbar_root):
    print(f"\n📁 {root}")
    for file in files:
        print("   ", file)


📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural
    dataset.json

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Vanilla_1shot
    statistics.json
    result.json

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Metrics_Reference
    statistics.json
    result.json

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Swap
    statistics.json
    result.json

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2/Vanilla_NoRules
    statistics.json
    r

In [20]:
import os

alpaca_root = "/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval"

for root, dirs, files in os.walk(alpaca_root):
    print(f"\n📁 {root}")
    for file in files:
        print("   ", file)


📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval
    requirements.txt
    setup.py
    CITATION.cff
    LICENSE
    .pre-commit-config.yaml
    README.md
    MANIFEST.in
    .gitignore
    pytest.ini

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval/.github

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval/.github/workflows
    integration_tests.yml
    test_update_leaderboard.yml
    set_version.py
    unit_tests.yml
    update_pypi.yml
    update_leaderboard.yml

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval/docs
    format_export_leaderboards.py
    AlpacaFarm_small.png
    check_unwanted_files.py
    format_sample_sheets.py
    index.html

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval/docs/data_AlpacaEval
    claude_leaderboard.csv
    chatgpt_fn_leaderboard.csv
    alpaca_eval_gpt4_leaderboard.csv

📁 /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/alpacaeval/docs/data_Alpac

In [21]:
import pandas as pd
from pathlib import Path

PROCESSED_DATA = PROJECT_ROOT / "datasets" / "processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

UNIFIED_COLUMNS = [
    "dataset",
    "subset",
    "sample_id",
    "instruction",
    "response_a",
    "response_b",
    "gold_label",
    "turn",
    "model_a",
    "model_b"
]

unified_df = pd.DataFrame(columns=UNIFIED_COLUMNS)

print(unified_df)

Empty DataFrame
Columns: [dataset, subset, sample_id, instruction, response_a, response_b, gold_label, turn, model_a, model_b]
Index: []


In [22]:
def convert_mtbench(mt_df):
    rows = []

    for _, row in mt_df.iterrows():

        conv_a = row["conversation_a"]
        conv_b = row["conversation_b"]

        # Extract user instruction
        instruction = ""
        for msg in conv_a:
            if msg.get("role") == "user":
                instruction = msg.get("content", "")
                break

        # Extract Assistant A response
        response_a = ""
        for msg in reversed(conv_a):
            if msg.get("role") == "assistant":
                response_a = msg.get("content", "")
                break

        # Extract Assistant B response
        response_b = ""
        for msg in reversed(conv_b):
            if msg.get("role") == "assistant":
                response_b = msg.get("content", "")
                break

        rows.append({
             "dataset": "MTBench",
    "subset": "GPT4_Pair",
    "sample_id": row["question_id"],
    "instruction": instruction,
    "response_a": response_a,
    "response_b": response_b,
    "gold_label": row["winner"],
    "turn": row["turn"],
    "model_a": row["model_a"],
    "model_b": row["model_b"]
            }
        )

    return pd.DataFrame(rows)

In [23]:
mtbench_unified = convert_mtbench(mt_df)

print(mtbench_unified.shape)
mtbench_unified.head()

(2400, 10)


,dataset,subset,sample_id,instruction,response_a,response_b,gold_label,turn,model_a,model_b
0,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Here is the travel blog post rewritten with ev...,model_b,1,alpaca-13b,claude-v1
1,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Here is the travel blog post rewritten with ev...,model_b,2,alpaca-13b,claude-v1
2,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Absolutely! A recent trip to the beautiful isl...,model_b,1,alpaca-13b,gpt-3.5-turbo
3,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Absolutely! A recent trip to the beautiful isl...,model_b,2,alpaca-13b,gpt-3.5-turbo
4,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,"Astonishing adventures awaited me in Hawaii, t...",model_b,1,alpaca-13b,gpt-4


In [24]:
!find /content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar -maxdepth 3

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/LLaMA2
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/ChatGPT
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/ChatGPT-0301
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/GPT-4
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/Falcon
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/evaluators/PaLM2
/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/dataset.json
/content/Prompt-Bias-

In [25]:
import json

path = "/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw/LLMBar/Dataset/LLMBar/Natural/dataset.json"

with open(path, "r") as f:
    data = json.load(f)

print(type(data))
print("Number of samples:", len(data))
print("\nKeys:")
print(data[0].keys())

print("\nFirst sample:")
print(data[0])

<class 'list'>
Number of samples: 100

Keys:
dict_keys(['input', 'output_1', 'output_2', 'label'])

First sample:
{'input': "Summarize the following content.\n\nMy girlfriend is Malaysian and has been studying in the UK for the past 3 years. We have been in a relationship for 2 ½ years now.\nHer visa to stay here is coming to an end later this year, then she will be forced to return to Malaysia despite opting to stay here if she had the choice. We have gone down the job route, to the point that she was offered the job here, but the employer failed to get a license to issue Visas on very petty grounds.\nI (and others) have suggested getting married. It is something we've discussed before, and we are both happy to do it except that she refuses to get married before she goes back as she feels like she will just be doing it so she can get a visa, rather than because she will actually be married. She's happy for me to propose in 6 months, but not before she returns. The problem for me is th

In [26]:
import json
import pandas as pd
from pathlib import Path

LLMBAR_ROOT = PROJECT_ROOT / "datasets" / "raw" / "LLMBar" / "Dataset" / "LLMBar"

def convert_llmbar():

    rows = []

    dataset_files = {
        "Natural": LLMBAR_ROOT / "Natural" / "dataset.json",
        "GPTInst": LLMBAR_ROOT / "Adversarial" / "GPTInst" / "dataset.json",
        "GPTOut": LLMBAR_ROOT / "Adversarial" / "GPTOut" / "dataset.json",
        "Manual": LLMBAR_ROOT / "Adversarial" / "Manual" / "dataset.json",
        "Neighbor": LLMBAR_ROOT / "Adversarial" / "Neighbor" / "dataset.json",
    }

    sample_id = 0

    for subset, filepath in dataset_files.items():

        with open(filepath, "r") as f:
            data = json.load(f)

        print(f"{subset}: {len(data)} samples")

        for sample in data:

            rows.append({

                "dataset": "LLMBar",
                "subset": subset,

                "sample_id": sample_id,

                "instruction": sample["input"],

                "response_a": sample["output_1"],

                "response_b": sample["output_2"],

                "gold_label": (
    "model_a"
    if sample["label"] == 1
    else "model_b"
),

                "turn": 1,

                "model_a": "Unknown",

                "model_b": "Unknown"

            })

            sample_id += 1

    return pd.DataFrame(rows)

In [27]:
llmbar_unified = convert_llmbar()

llmbar_unified.head()

Natural: 100 samples
GPTInst: 92 samples
GPTOut: 47 samples
Manual: 46 samples
Neighbor: 134 samples


,dataset,subset,sample_id,instruction,response_a,response_b,gold_label,turn,model_a,model_b
0,LLMBar,Natural,0,Summarize the following content.\n\nMy girlfri...,My girlfriend's visa to stay in the UK expires...,My girlfriend is Malaysian and has been studyi...,model_a,1,Unknown,Unknown
1,LLMBar,Natural,1,Complete a brief story given the following fir...,He learned his weather report. He prepared for...,He would always mistreat it every day. He eats...,model_a,1,Unknown,Unknown
2,LLMBar,Natural,2,How many integers are in the solution of the i...,The inequality |x + 5| < 10 means that the abs...,The list of integers that satisfy the inequali...,model_a,1,Unknown,Unknown
3,LLMBar,Natural,3,Why does eating something crunchy sound so lou...,The sound is transmitted through the vibration...,The noise that crunchy foods make when we eat ...,model_a,1,Unknown,Unknown
4,LLMBar,Natural,4,Summarize the following content.\n\nMy bf only...,Boyfriend only likes to talk through text. He ...,My boyfriend only likes to text me for shallow...,model_b,1,Unknown,Unknown


In [38]:
unified_df = pd.concat(
    [mtbench_unified, llmbar_unified],
    ignore_index=True
)

print("Unified Dataset Shape:", unified_df.shape)

unified_df.head()

Unified Dataset Shape: (2819, 10)


,dataset,subset,sample_id,instruction,response_a,response_b,gold_label,turn,model_a,model_b
0,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Here is the travel blog post rewritten with ev...,model_b,1,alpaca-13b,claude-v1
1,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Here is the travel blog post rewritten with ev...,model_b,2,alpaca-13b,claude-v1
2,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Absolutely! A recent trip to the beautiful isl...,model_b,1,alpaca-13b,gpt-3.5-turbo
3,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,Absolutely! A recent trip to the beautiful isl...,model_b,2,alpaca-13b,gpt-3.5-turbo
4,MTBench,GPT4_Pair,81,Compose an engaging travel blog post about a r...,Aloha! I recently had the pleasure of visiting...,"Astonishing adventures awaited me in Hawaii, t...",model_b,1,alpaca-13b,gpt-4


In [39]:
unified_df.to_csv(
    PROCESSED_DATA / "unified_dataset.csv",
    index=False
)
print("✅ Unified dataset saved.")

✅ Unified dataset saved.


In [40]:
print(unified_df["dataset"].value_counts())

print()

print(unified_df["subset"].value_counts())

dataset
MTBench    2400
LLMBar      419
Name: count, dtype: int64

subset
GPT4_Pair    2400
Neighbor      134
Natural       100
GPTInst        92
GPTOut         47
Manual         46
Name: count, dtype: int64


In [43]:
!pwd

/content/Prompt-Bias-Mitigation-SLM-Judge/datasets/raw


In [44]:
%cd /content/Prompt-Bias-Mitigation-SLM-Judge

/content/Prompt-Bias-Mitigation-SLM-Judge


In [46]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	datasets/

nothing added to commit but untracked files present (use "git add" to track)


In [47]:
!find datasets -maxdepth 3

datasets
datasets/processed
datasets/processed/unified_dataset.csv
datasets/raw
datasets/raw/LLMBar
datasets/raw/LLMBar/requirements.txt
datasets/raw/LLMBar/.git
datasets/raw/LLMBar/LICENSE
datasets/raw/LLMBar/LLMEvaluator
datasets/raw/LLMBar/README.md
datasets/raw/LLMBar/.gitignore
datasets/raw/LLMBar/Dataset
datasets/raw/alpacaeval
datasets/raw/alpacaeval/.github
datasets/raw/alpacaeval/docs
datasets/raw/alpacaeval/tests
datasets/raw/alpacaeval/results
datasets/raw/alpacaeval/example
datasets/raw/alpacaeval/requirements.txt
datasets/raw/alpacaeval/src
datasets/raw/alpacaeval/setup.py
datasets/raw/alpacaeval/.git
datasets/raw/alpacaeval/CITATION.cff
datasets/raw/alpacaeval/LICENSE
datasets/raw/alpacaeval/.pre-commit-config.yaml
datasets/raw/alpacaeval/README.md
datasets/raw/alpacaeval/MANIFEST.in
datasets/raw/alpacaeval/client_configs
datasets/raw/alpacaeval/.gitignore
datasets/raw/alpacaeval/scripts
datasets/raw/alpacaeval/notebooks
datasets/raw/alpacaeval/pytest.ini
datasets/raw/alp